In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_14/data'
print(os.listdir(data_dir))

['payments.csv', 'claims.csv', 'policies.csv']


In [2]:
import pandas as pd
payments = pd.read_csv(os.path.join(data_dir, 'payments.csv'))
claims = pd.read_csv(os.path.join(data_dir, 'claims.csv'))
policies = pd.read_csv(os.path.join(data_dir, 'policies.csv'))

print("Payments columns:", payments.columns)
print("Claims columns:", claims.columns)
print("Policies columns:", policies.columns)


Payments columns: Index(['payment_id', 'claim_id', 'payment_date', 'amount_paid'], dtype='str')
Claims columns: Index(['claim_id', 'policy_id', 'loss_date', 'report_date', 'close_date',
       'cause', 'adjuster', 'claim_amount'],
      dtype='str')
Policies columns: Index(['policy_id', 'product', 'region', 'premium_monthly', 'deductible',
       'start_date'],
      dtype='str')


In [3]:
# Calculate total payments per claim
total_payments = payments.groupby('claim_id')['amount_paid'].sum().reset_index()
total_payments.rename(columns={'amount_paid': 'total_paid'}, inplace=True)

# Find last payment date per claim
payments['payment_date'] = pd.to_datetime(payments['payment_date'])
last_payment = payments.groupby('claim_id')['payment_date'].max().reset_index()
last_payment.rename(columns={'payment_date': 'last_payment_date'}, inplace=True)

# Merge claims, policies, total_payments, and last_payment
df = claims.merge(policies[['policy_id', 'deductible']], on='policy_id', how='left')
df = df.merge(total_payments, on='claim_id', how='left')
df = df.merge(last_payment, on='claim_id', how='left')

# Filter claims where total payments > deductible
df_filtered = df[df['total_paid'] > df['deductible']].copy()

# Calculate settlement lag
df_filtered['loss_date'] = pd.to_datetime(df_filtered['loss_date'])
df_filtered['settlement_lag'] = (df_filtered['last_payment_date'] - df_filtered['loss_date']).dt.days

# Group by cause and calculate median settlement lag
median_lag = df_filtered.groupby('cause')['settlement_lag'].median().reset_index()

# Sort by median lag and then alphabetically by cause
median_lag = median_lag.sort_values(by=['settlement_lag', 'cause'])

print(median_lag.head())


       cause  settlement_lag
0  Collision             8.0
1      Theft             8.0
2      Water             8.0
3    Weather             8.5


In [4]:
print(median_lag)

       cause  settlement_lag
0  Collision             8.0
1      Theft             8.0
2      Water             8.0
3    Weather             8.5
